# Random Forest Component: Personal Loan Prediction

This notebook explains the Random Forest workflow for the bank personal loan dataset in a report-friendly order: data understanding, preprocessing, training, tuning, evaluation, and interpretation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, average_precision_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.model_selection import GridSearchCV, train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("dataset/Bank_Personal_Loan_Modelling.csv")
display(df.head())
print(df.shape)
print(df["Personal Loan"].value_counts())

In [ ]:
df = df.copy()
df["Experience"] = df["Experience"].clip(lower=0)
X = df.drop(columns=["Personal Loan", "ID", "ZIP Code"])
y = df["Personal Loan"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
param_grid = {
    "n_estimators": [200],
    "max_depth": [None, 12],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=1), param_grid, cv=5, scoring="f1", n_jobs=1)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_
print(grid.best_params_)

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
    "Average Precision": average_precision_score(y_test, y_proba),
}
display(pd.DataFrame(metrics.items(), columns=["Metric", "Value"]))
print(classification_report(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Greens")
plt.title("Confusion Matrix - Random Forest")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve - Random Forest")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.title("Precision-Recall Curve - Random Forest")
plt.show()

In [ ]:
importance_df = pd.DataFrame({"feature": X.columns, "importance": best_model.feature_importances_}).sort_values("importance", ascending=False)
sns.barplot(data=importance_df.head(10), x="importance", y="feature", hue="feature", dodge=False, legend=False, palette="viridis")
plt.title("Top 10 Feature Importances - Random Forest")
plt.show()